In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 데이터셋 준비 (NumPy 데이터를 PyTorch Tensor로 변환)
# 입력 데이터 (9개 샘플, 2개 특성)
X = torch.tensor([
    [1.0, 2.0], [1.5, 1.8], [0.8, 2.5],  # Class 0
    [8.0, 8.0], [7.5, 9.0], [8.5, 7.8],  # Class 1
    [1.0, 8.0], [1.2, 9.0], [0.5, 8.5]   # Class 2
], dtype=torch.float32)

# 타겟 정답 (PyTorch CrossEntropyLoss는 원-핫 인코딩이 아닌 클래스 인덱스(0, 1, 2)를 전달받습니다)
y = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2], dtype=torch.long)


# 2. 신경망 모델 정의 (nn.Module 상속)
class MultiClassNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MultiClassNet, self).__init__()
        # 계층 정의
        self.fc1 = nn.Linear(input_size, hidden_size)  # 입력층 -> 은닉층
        self.sigmoid = nn.Sigmoid()                    # 활성화 함수
        self.fc2 = nn.Linear(hidden_size, output_size) # 은닉층 -> 출력층

    def forward(self, x):
        out = self.fc1(x)
        out = self.sigmoid(out)
        out = self.fc2(out)  # Softmax는 nn.CrossEntropyLoss 내부에서 처리되므로 생략
        return out

# 3. 모델, 손실 함수, 옵티마이저 생성
input_size = 2
hidden_size = 5
output_size = 3
learning_rate = 0.5

# 재현성을 위한 시드 고정
torch.manual_seed(42)

model = MultiClassNet(input_size, hidden_size, output_size)

# 손실 함수: Softmax + Cross-Entropy가 결합된 형태
criterion = nn.CrossEntropyLoss()

# 최적화 알고리즘: 경사하강법(SGD)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)


# 4. 모델 학습 루프 (Training Loop)
print("=== PyTorch 학습 시작 ===")
epochs = 3000

for epoch in range(epochs):
    # ① 순전파 (Forward)
    outputs = model(X)
    loss = criterion(outputs, y)

    # ② 역전파 (Backward) 및 가중치 업데이트
    optimizer.zero_grad()  # 이전 스텝의 기울기(Gradient) 초기화
    loss.backward()        # 자동 미분을 통해 역전파 수행 (autograd)
    optimizer.step()       # 경사하강법으로 가중치 업데이트

    # 출력
    if (epoch + 1) % 500 == 0:
        # 가장 높은 확률 값을 가진 클래스 인덱스 추출
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y).float().mean() * 100
        print(f"Epoch {epoch + 1:4d} | Loss: {loss.item():.4f} | Accuracy: {accuracy.item():.1f}%")


# 5. 테스트 샘플 예측
print("\n=== 테스트 샘플 예측 ===")
model.eval() # 평가 모드 전환
test_sample = torch.tensor([[1.2, 2.1], [8.1, 8.2], [0.9, 8.8]], dtype=torch.float32)

with torch.no_grad(): # 테스트 단계에서는 기울기 계산 불필요
    logits = model(test_sample)
    # 실제 확률 분포를 보고 싶다면 torch.softmax 적용
    probabilities = torch.softmax(logits, dim=1)
    predictions = torch.argmax(probabilities, dim=1)

for i, (prob, pred) in enumerate(zip(probabilities, predictions)):
    prob_list = [round(p, 3) for p in prob.tolist()]
    print(f"샘플 {i+1} 확률 분포: {prob_list} -> 최종 예측 클래스: Class {pred.item()}")

=== PyTorch 학습 시작 ===
Epoch  500 | Loss: 0.0097 | Accuracy: 100.0%
Epoch 1000 | Loss: 0.0043 | Accuracy: 100.0%
Epoch 1500 | Loss: 0.0027 | Accuracy: 100.0%
Epoch 2000 | Loss: 0.0020 | Accuracy: 100.0%
Epoch 2500 | Loss: 0.0016 | Accuracy: 100.0%
Epoch 3000 | Loss: 0.0013 | Accuracy: 100.0%

=== 테스트 샘플 예측 ===
샘플 1 확률 분포: [0.999, 0.001, 0.0] -> 최종 예측 클래스: Class 0
샘플 2 확률 분포: [0.001, 0.999, 0.001] -> 최종 예측 클래스: Class 1
샘플 3 확률 분포: [0.001, 0.001, 0.999] -> 최종 예측 클래스: Class 2
